# COMP6713 — Notebook 2: TF-IDF Baseline

The baseline model uses **TF-IDF vectorisation + cosine similarity** to retrieve
the most relevant document for a given question, then extracts the single most
relevant sentence as the answer.

This intentionally simple approach establishes a lower-bound score that all
more sophisticated models should exceed.

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
from medqa.data.loader import load_all
from medqa.data.preprocessor import split_dataset
from medqa.models.baseline import TFIDFBaseline

## 1. Load & Split Data

In [ ]:
records = load_all()
train, test = split_dataset(records)
print(f"Train: {len(train)} | Test: {len(test)}")

## 2. Build TF-IDF Index

In [ ]:
model = TFIDFBaseline()
# Try loading saved index first; build if not found
if not model.load():
    model.fit(train)
    print("Index built and saved.")
else:
    print("Index loaded from disk.")
print(f"Vocabulary size: {len(model.vectorizer.vocabulary_):,}")

## 3. Single Question Demo

In [ ]:
question = "What is the relationship between obesity and hypertension?"
result = model.predict(question)

print(f"Question  : {question}")
print(f"Answer    : {result['predicted_answer']}")
print(f"Score     : {result['score']:.4f}")

## 4. Batch Evaluation on Test Set

In [ ]:
import nltk
nltk.download('punkt_tab', quiet=True)

from medqa.evaluation.metrics import evaluate, print_results

test_subset = test[:200]
questions   = [r['question'] for r in test_subset]
golds       = [r['answer']   for r in test_subset]

preds = [model.predict(q)['predicted_answer'] for q in questions]
results = evaluate(preds, golds, use_bertscore=False)
print_results(results, model_name='TF-IDF Baseline')

## 5. Qualitative Error Analysis

In [ ]:
from medqa.evaluation.qualitative import analyse_errors
from medqa.evaluation.metrics import token_f1

eval_records = []
for r, pred in zip(test_subset, preds):
    eval_records.append({
        'question':         r['question'],
        'predicted_answer': pred,
        'gold_answer':      r['answer'],
        'context':          r['context'],
        'token_f1':         token_f1(pred, r['answer']),
    })

summary = analyse_errors(eval_records, f1_threshold=0.3, output_path='../data/processed/baseline_errors.json')

## 6. Retrieval Examples — Good vs Bad

In [ ]:
import random
random.seed(42)
good = [e for e in eval_records if e['token_f1'] > 0.4][:2]
bad  = [e for e in eval_records if e['token_f1'] == 0.0][:2]

for label, examples in [('GOOD', good), ('BAD', bad)]:
    print(f"\n{'='*60}")
    print(f"  {label} EXAMPLES")
    print(f"{'='*60}")
    for e in examples:
        print(f"  Q  : {e['question'][:100]}")
        print(f"  Pred: {e['predicted_answer'][:100]}")
        print(f"  Gold: {e['gold_answer'][:100]}")
        print(f"  F1  : {e['token_f1']:.3f}")
        print()

## Summary

The TF-IDF baseline is **intentionally simple** and serves as a lower bound.

**Expected results (200 test samples):**
- Exact Match ≈ 0.00 (long-form answers rarely match exactly)
- Token F1 ≈ 0.03–0.05
- BERTScore ≈ 0.78 (semantic similarity is reasonable)

The main failure mode is **PARTIAL_ANSWER** — the model retrieves a related
document but the extracted sentence is not the gold answer. This motivates
the use of fine-tuned models and RAG in subsequent notebooks.